# ANN implementation from Scratch

In [31]:
import numpy as np

In [32]:
#Activation Functions
def sigmoid(x):
  """1 / (1 + e^-x)"""
  return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def relu(x):
  """max(0, x)"""
  return np.maximum(0, x)

def tanh(x):
  return np.tanh(x)

def linear(x):
  return x

#Activation Derivatives
def sigmoid_derivative(x):
  """f(x) * (1 - f(x))"""
  s = sigmoid(x)
  return s * (1 - s)

def relu_derivative(x):
  """1 if x > 0 else 0"""
  return np.where(x > 0, 1.0, 0.0)

def tanh_derivative(x):
  """1 - tanh^2(x)"""
  return 1.0 - np.tanh(x) ** 2

def linear_derivative(x):
  return np.ones_like(x)

#Loss Functions
def mse(y_true, y_pred):
  return np.mean((y_pred - y_true) ** 2)

def mae(y_true, y_pred):
  return np.mean(np.abs(y_pred - y_true))

#Loss Derivatives with respect to y_pred
def mse_derivative(y_true, y_pred):
  """d(MSE)/dy_pred = 2 * (y_pred - y_true) / N"""
  n = y_true.shape[0]
  return 2 * (y_pred - y_true) / n

def mae_derivative(y_true, y_pred):
  """d(MAE)/dy_pred = sign(y_pred - y_true) / N"""
  n = y_true.shape[0]
  return np.sign(y_pred - y_true) / n

# Notes

## For each Dense layer

**Weights matrix:** shape (input_size × output_size).
Column j holds all incoming weights of neuron j. Row i holds all weights leaving input feature i.

**Bias matrix:** shape (1 × output_size).
One bias per neuron. Broadcast across every row (sample) when added.

**Input matrix:** shape (batch_size × input_size).
Each row is a sample, each column is a feature.

**self.z:** shape (batch_size × output_size).
Each row is a sample, each column is a neuron.

**self.inputs and self.z** are stored during `forward` so `backward` can use them.

## Forward

    z = inputs @ weights + bias
    (batch × in) @ (in × out) → (batch × out)
    bias (1 × out) is broadcast over rows

## Backward

Every gradient has the same shape as the thing it is the gradient of:

    dZ              = output_gradient * activation_derivative(z)   (batch × out)
    weights_gradient = inputs.T @ dZ                               (in × out)
    bias_gradient    = sum of dZ over rows (axis=0)                (1 × out)
    input_gradient   = dZ @ weights.T                              (batch × in)

`input_gradient` is returned and becomes the previous layer's `output_gradient`.

In [33]:
class Dense:
  def __init__(self, input_size, output_size, activation=None, activation_derivative=None):
    # Standard Xavier/He initialization for row-based samples (input_size x output_size)
    self.weights = np.random.randn(input_size, output_size) * np.sqrt(2.0 / input_size)
    self.bias = np.zeros((1, output_size))
    self.activation = activation
    self.activation_derivative = activation_derivative
    self.inputs = None
    self.z = None

  def forward(self, inputs):
    # inputs shape: (batch_size, input_size)
    self.inputs = inputs
    self.z = inputs @ self.weights + self.bias

    if self.activation:
      return self.activation(self.z)
    return self.z

  def backward(self, output_gradient, learning_rate):
    # output_gradient shape: (batch_size, output_size)
    if self.activation_derivative:
      output_gradient = output_gradient * self.activation_derivative(self.z)

    # Calculate gradients
    weights_gradient = self.inputs.T @ output_gradient
    bias_gradient = np.sum(output_gradient, axis=0, keepdims=True)

    # Gradient to propagate backward to the previous layer
    input_gradient = output_gradient @ self.weights.T

    # Update parameters
    self.weights -= learning_rate * weights_gradient
    self.bias -= learning_rate * bias_gradient

    return input_gradient

In [34]:
class ANN:
  def __init__(self, layers=None, loss=mse, loss_derivative=mse_derivative):
    self.layers = [] if layers is None else layers
    self.loss = loss
    self.loss_derivative = loss_derivative

  def forward(self, inputs):
    for layer in self.layers:
      inputs = layer.forward(inputs)
    return inputs

  def train(self, X, Y, epochs, learning_rate):
    # X: input data, Y: target values
    for epoch in range(epochs):
      # Forward propagation
      output = self.forward(X)

      # Compute loss and loss gradient dynamically using configured functions
      current_loss = self.loss(Y, output)
      loss_gradient = self.loss_derivative(Y, output)

      # Print loss progress every 1000 epochs
      if epoch % 1000 == 0:
        print(f"Epoch {epoch:4d} - Loss: {current_loss:.6f}")

      # Backward propagation
      gradient = loss_gradient
      for layer in reversed(self.layers):
        gradient = layer.backward(gradient, learning_rate)

  def predict(self, inputs):
    return self.forward(inputs)

In [35]:
# Re-seed to ensure reproducibility
np.random.seed(42)

# XOR dataset: Rows are samples
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
Y = np.array([[0], [1], [1], [0]])

# Instantiate ANN using 4 hidden neurons
network = ANN([
    Dense(2, 4, activation=tanh, activation_derivative=tanh_derivative),
    Dense(4, 1, activation=sigmoid, activation_derivative=sigmoid_derivative)
])

# training
epochs = 10000
learning_rate = 0.1

print("Training the row-oriented ANN on XOR (Hidden Dim: 4)...")
network.train(X, Y, epochs, learning_rate)

predictions = network.predict(X)
print("\nFinal Results:")
for i in range(4):
    print(f"Input: {X[i]} -> Predicted: {predictions[i, 0]:.4f} (Expected: {Y[i, 0]})")

Training the row-oriented ANN on XOR (Hidden Dim: 4)...
Epoch    0 - Loss: 0.294174
Epoch 1000 - Loss: 0.081010
Epoch 2000 - Loss: 0.011159
Epoch 3000 - Loss: 0.004831
Epoch 4000 - Loss: 0.002961
Epoch 5000 - Loss: 0.002104
Epoch 6000 - Loss: 0.001620
Epoch 7000 - Loss: 0.001312
Epoch 8000 - Loss: 0.001100
Epoch 9000 - Loss: 0.000945

Final Results:
Input: [0 0] -> Predicted: 0.0128 (Expected: 0)
Input: [0 1] -> Predicted: 0.9706 (Expected: 1)
Input: [1 0] -> Predicted: 0.9696 (Expected: 1)
Input: [1 1] -> Predicted: 0.0368 (Expected: 0)


In [38]:
# Re-seed to ensure reproducibility
np.random.seed(42)

# OR dataset: Rows are samples
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
Y = np.array([[0], [1], [1], [1]])

# Instantiate ANN using 2 hidden neurons
network = ANN([
    Dense(2, 5, activation=tanh, activation_derivative=tanh_derivative),
    Dense(5, 2, activation=tanh, activation_derivative=tanh_derivative),
    Dense(2, 1, activation=sigmoid, activation_derivative=sigmoid_derivative)
])

# training
epochs = 10000
learning_rate = 0.1

print("Training the row-oriented ANN on OR (Hidden Dim: 2)...")
network.train(X, Y, epochs, learning_rate)

predictions = network.predict(X)
print("\nFinal Results:")
for i in range(4):
    print(f"Input: {X[i]} -> Predicted: {predictions[i, 0]:.4f} (Expected: {Y[i, 0]})")

Training the row-oriented ANN on OR (Hidden Dim: 2)...
Epoch    0 - Loss: 0.453324
Epoch 1000 - Loss: 0.002031
Epoch 2000 - Loss: 0.000848
Epoch 3000 - Loss: 0.000527
Epoch 4000 - Loss: 0.000380
Epoch 5000 - Loss: 0.000296
Epoch 6000 - Loss: 0.000242
Epoch 7000 - Loss: 0.000205
Epoch 8000 - Loss: 0.000177
Epoch 9000 - Loss: 0.000156

Final Results:
Input: [0 0] -> Predicted: 0.0171 (Expected: 0)
Input: [0 1] -> Predicted: 0.9894 (Expected: 1)
Input: [1 0] -> Predicted: 0.9897 (Expected: 1)
Input: [1 1] -> Predicted: 0.9932 (Expected: 1)


In [39]:
import numpy as np
# Re-seed to ensure reproducibility
np.random.seed(42)

# AND dataset: Rows are samples
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
Y = np.array([[0], [0], [0], [1]])

# Instantiate ANN using 2 hidden neurons
network = ANN([
    Dense(2, 7, activation=tanh, activation_derivative=tanh_derivative),
    Dense(7, 1, activation=sigmoid, activation_derivative=sigmoid_derivative)
])

# training
epochs = 10000
learning_rate = 0.1

print("Training the row-oriented ANN on AND (Hidden Dim: 2)...")
network.train(X, Y, epochs, learning_rate)

predictions = network.predict(X)
print("\nFinal Results:")
for i in range(4):
    print(f"Input: {X[i]} -> Predicted: {predictions[i, 0]:.4f} (Expected: {Y[i, 0]})")

Training the row-oriented ANN on AND (Hidden Dim: 2)...
Epoch    0 - Loss: 0.291695
Epoch 1000 - Loss: 0.005554
Epoch 2000 - Loss: 0.001994
Epoch 3000 - Loss: 0.001139
Epoch 4000 - Loss: 0.000778
Epoch 5000 - Loss: 0.000583
Epoch 6000 - Loss: 0.000462
Epoch 7000 - Loss: 0.000381
Epoch 8000 - Loss: 0.000323
Epoch 9000 - Loss: 0.000280

Final Results:
Input: [0 0] -> Predicted: 0.0003 (Expected: 0)
Input: [0 1] -> Predicted: 0.0144 (Expected: 0)
Input: [1 0] -> Predicted: 0.0184 (Expected: 0)
Input: [1 1] -> Predicted: 0.9791 (Expected: 1)
